# Lab 1: Vectorless RAG


## Setup

### Install Dependencies

In [20]:
!pip install -q pageindex pymupdf langchain-aws langchain-core boto3

### Import Libraries


In [21]:
import os       # for environment variables
import json     # for parsing LLM JSON responses
import pymupdf  # for PDF text extraction
from langchain_aws import ChatBedrockConverse  # LangChain AWS Bedrock client
from langchain_core.messages import HumanMessage  # typed message objects

### Configure AWS Bedrock Credentials

In [ ]:
os.environ["AWS_ACCESS_KEY_ID"]     = "YOUR_ACCESS_KEY_ID"
os.environ["AWS_SECRET_ACCESS_KEY"] = "YOUR_SECRET_ACCESS_KEY"
os.environ["AWS_ENDPOINT_URL"]      = "https://api.enterprisesi.co/api/v1/aws-genai/bedrock-runtime"
os.environ["AWS_REGION"]            = "ap-south-1"

print("Credentials configured.")

### Load PageIndex API Key

In [ ]:
PAGEINDEX_API_KEY = input("Enter your PageIndex API key (get one at https://pageindex.ai): ").strip()
os.environ["PAGEINDEX_API_KEY"] = PAGEINDEX_API_KEY

print("PageIndex key loaded.")

### Set up the LLM

In [ ]:
def call_llm(prompt, model="global.amazon.nova-2-lite-v1:0"):
    llm = ChatBedrockConverse(
        model=model,
        temperature=0,
        max_tokens=512,
    )
    response = llm.invoke([HumanMessage(content=prompt)])
    return response.content.strip()

---
## Step 1 — Build Document Tree


### Build Document Tree


In [ ]:
# PageIndex SDK for document tree generation and utilities
from pageindex import PageIndexClient
from pageindex import utils
import time

# Path to the PDF file
PDF_PATH = "data/synthetic_medicare_plus_policy_detailed.pdf"

# Submit PDF to PageIndex for tree generation
pi = PageIndexClient(api_key=PAGEINDEX_API_KEY)
result = pi.submit_document(PDF_PATH)
doc_id = result["doc_id"]
print(f"Submitted: {doc_id}")

# Poll until processing completes (max 5 min)
elapsed = 0
while elapsed < 300:
    if pi.is_retrieval_ready(doc_id):
        break
    time.sleep(5)
    elapsed += 5
    print(f"  {elapsed}s...")
else:
    raise TimeoutError("PageIndex timeout")

# Retrieve the hierarchical tree (titles + summaries, no full text)
tree = pi.get_tree(doc_id, node_summary=True)["result"]
utils.print_tree(tree, exclude_fields=["text"])

Submitted: pi-cmrm9m8n900iz01o4iqososuo
  5s...
  10s...
  15s...
[{'title': 'Century Communities Reports First Quarte...',
  'node_id': '0000',
  'page_index': 1,
  'summary': 'Century Communities reported its Q1 2025...'},
 {'title': 'First Quarter 2025 Results',
  'node_id': '0001',
  'page_index': 1,
  'prefix_summary': 'The report details the Q1 2025 financial...',
  'nodes': [{'title': 'Balance Sheet and Liquidity',
             'node_id': '0002',
             'page_index': 2,
             'summary': 'In Q1 2025, the company maintained a str...'},
            {'title': 'Full Year 2025 Outlook',
             'node_id': '0003',
             'page_index': 2,
             'summary': '## Full Year 2025 Outlook\n\nScott Dixon, ...'},
            {'title': 'Webcast and Conference Call',
             'node_id': '0004',
             'page_index': 2,
             'summary': 'Century Communities will host a webcast ...'},
            {'title': 'About Century Communities',
             'node

---
## Step 2 — Ask a Question

In [26]:
QUERY = "What was the total revenue reported in the earnings release?"

---
## Step 3 — LLM Finds Relevant Sections

The LLM reads the tree (titles + summaries only — no full text) and picks which nodes likely contain the answer.

### Search the Tree


In [49]:
# Strip full text from tree — LLM only needs titles + summaries to pick relevant nodes
tree_slim = utils.remove_fields(tree.copy(), fields=["text"])

# Ask the LLM which nodes are relevant to the question
# We use JSON format so we can reliably extract structured data
search_prompt = f"""
IMPORTANT: You MUST respond with valid JSON only. No other text.

You are given a question and a document tree.
Each node has: node_id, title, summary.
Find all nodes likely to contain the answer.

Question: {QUERY}

Document tree:
{json.dumps(tree_slim, indent=2)}

Respond with ONLY this JSON format:
{{
    "thinking": "<your reasoning>",
    "node_list": ["node_id_1", "node_id_2"]
}}
"""

# Try to parse JSON — handle cases where LLM adds extra text or returns invalid JSON
import re
response = call_llm(search_prompt)
try:
    result = json.loads(response)
except json.JSONDecodeError:
    # Fallback: try to extract JSON from the response using regex
    match = re.search(r'\{.*\}', response, re.DOTALL)
    if match:
        result = json.loads(match.group())
    else:
        print("Could not parse LLM response. Using empty result.")
        result = {"thinking": "", "node_list": []}

### Map Nodes to Page Numbers

The LLM returns node IDs (e.g. `0000`, `0001`), but we need to know which **pages** those nodes correspond to. This step is required because the next cell extracts text from PDF pages — and it needs page numbers, not node IDs.

In [50]:
# Map node IDs to their metadata (title, page range, etc.)
node_map = utils.create_node_mapping(tree, include_page_ranges=True, max_page=len(page_texts))

# Display the LLM's reasoning and which nodes it selected
print("\nReasoning:", result.get("thinking", ""), "\n")
print("Retrieved nodes:")
for nid in result.get("node_list", []):
    if nid in node_map:
        info = node_map[nid]
        # Show single page number or range (e.g. "3" or "3-5")
        pages = info['start_index'] if info['start_index'] == info['end_index'] else f"{info['start_index']}-{info['end_index']}"
        print(f"  {nid} | Pages {pages} | {info['node']['title']}")
    else:
        print(f"  {nid} | not found in tree")


Reasoning: The question asks for the total revenue reported in the earnings release. The root node (0000) summary states '$903.2 million in revenue'. The child node (0001) prefix_summary states 'total revenues of $903.2M from 2,284 home deliveries'. Both nodes contain the revenue figure, so both are relevant. 

Retrieved nodes:
  0000 | Pages 1 | Century Communities Reports First Quarter 2025 Results
  0001 | Pages 1-2 | First Quarter 2025 Results


---
## Step 4 — LLM Answers from Extracted Text

### Extract Text from PDF

Extract text from each page of the PDF using PyMuPDF.

In [51]:
doc = pymupdf.open(PDF_PATH)
# len(doc) = total pages; i goes from 0 to len(doc)-1
# i+1 makes page numbers 1-based (page 1, 2, 3...)
# doc.load_page(i).get_text() extracts raw text from page i
page_texts = {i+1: doc.load_page(i).get_text() for i in range(len(doc))}
doc.close()
print(f"Extracted text from {len(page_texts)} pages.")

Extracted text from 11 pages.


### Build Context

In [52]:
# Collect text from pages covered by retrieved nodes (deduplicating pages)
texts, seen = [], set()
for nid in result.get("node_list", []):
    if nid not in node_map:
        continue
    info = node_map[nid]
    # Loop through each page in the node's range
    for p in range(info["start_index"], info["end_index"] + 1):
        # Only add if we haven't seen this page and it has text
        if p not in seen and p in page_texts:
            texts.append(f"--- Page {p} ---\n{page_texts[p]}")
            seen.add(p)
# Join all extracted text into one context string
context = "\n\n".join(texts)
print(f"Using {len(context.splitlines())} lines of text.")

Using 89 lines of text.


### Generate Answer


In [53]:
# Send the extracted text + question to the LLM for the final answer.
# The prompt tells the LLM to only use the provided context.
system_prompt = f"""
Answer the question based on the provided text.

Context:
{context}

Question: {QUERY}

Rules:
- Answer only from the context
- If the answer isn't there, say so
"""

answer = call_llm(system_prompt)
print(answer)

$903.2 million


---
## Try It Yourself

Change `QUERY` above and re-run from **Step 3**.